# Kittangle – звуки (Stable Audio 3.0)
Runtime → Change runtime type → **T4 GPU**. Затем Runtime → Run all. В ячейке 2 вставьте свой HF-токен (Read).

In [ ]:
!nvidia-smi -L
!pip -q install git+https://github.com/Stability-AI/stable-audio-3.git transformers soundfile librosa

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
SOUNDS = [
 {
  "id": "S01",
  "name": "ui_click_1",
  "music": false,
  "dur": 0.3,
  "prompt": "soft wooden UI button click, short, clean, cozy mobile game, no reverb"
 },
 {
  "id": "S02",
  "name": "ui_click_2",
  "music": false,
  "dur": 0.3,
  "prompt": "soft bubbly pop button tap, short, cute mobile game interface"
 },
 {
  "id": "S03",
  "name": "meow_1",
  "music": false,
  "dur": 1.0,
  "prompt": "cute short kitten meow, happy, high pitched, close microphone, clean studio recording, no background noise"
 },
 {
  "id": "S04",
  "name": "meow_2",
  "music": false,
  "dur": 1.0,
  "prompt": "short friendly cat meow, rising tone, playful, close microphone, clean, no background noise"
 },
 {
  "id": "S05",
  "name": "meow_3",
  "music": false,
  "dur": 1.0,
  "prompt": "tiny chirpy cat trill meow, cheerful, close microphone, clean, no background noise"
 },
 {
  "id": "S06",
  "name": "meow_4",
  "music": false,
  "dur": 1.0,
  "prompt": "short soft cat mew, content, close microphone, clean, no background noise"
 },
 {
  "id": "S07",
  "name": "purr",
  "music": false,
  "dur": 2.0,
  "prompt": "close cat purring, warm and soft, steady, clean recording, no background noise"
 },
 {
  "id": "S08",
  "name": "wake",
  "music": false,
  "dur": 1.5,
  "prompt": "cat yawning softly then a small sleepy mew, cute, clean recording"
 },
 {
  "id": "S09",
  "name": "bonk_1",
  "music": false,
  "dur": 0.8,
  "prompt": "soft cartoon bump into a cushion followed by a short annoyed cat grumble, cute, not violent"
 },
 {
  "id": "S10",
  "name": "bonk_2",
  "music": false,
  "dur": 0.8,
  "prompt": "light comedic boing thud and a tiny offended cat hiss, cute cartoon, short"
 },
 {
  "id": "S11",
  "name": "slide_1",
  "music": false,
  "dur": 0.8,
  "prompt": "quick soft swoosh of fur sliding on a wooden floor, light whoosh, short"
 },
 {
  "id": "S12",
  "name": "slide_2",
  "music": false,
  "dur": 0.8,
  "prompt": "gentle fast whoosh with a faint fabric rustle, short, clean"
 },
 {
  "id": "S13",
  "name": "coin",
  "music": false,
  "dur": 0.6,
  "prompt": "small sparkly pickup chime, bright, short, cute mobile game coin"
 },
 {
  "id": "S14",
  "name": "combo",
  "music": false,
  "dur": 0.6,
  "prompt": "short rising magical ding, bright bell, cute mobile game combo"
 },
 {
  "id": "S15",
  "name": "star",
  "music": false,
  "dur": 1.2,
  "prompt": "twinkling star reward sound, magical chime glissando up, short, cheerful"
 },
 {
  "id": "S16",
  "name": "task",
  "music": false,
  "dur": 1.2,
  "prompt": "cozy task complete sound, soft wooden thunk and sparkle, satisfying, short"
 },
 {
  "id": "S17",
  "name": "tips",
  "music": false,
  "dur": 1.2,
  "prompt": "coins pouring into a small glass jar, cheerful clink, short"
 },
 {
  "id": "S18",
  "name": "chest",
  "music": false,
  "dur": 2.0,
  "prompt": "treasure chest opening with a wooden creak and a magical sparkle burst, cute game reward"
 },
 {
  "id": "S19",
  "name": "scene",
  "music": false,
  "dur": 1.0,
  "prompt": "soft page turn with a gentle airy whoosh, storybook, short"
 },
 {
  "id": "S20",
  "name": "win",
  "music": false,
  "dur": 2.5,
  "prompt": "short happy victory jingle, marimba and glockenspiel, cozy cute puzzle game, ends cleanly"
 },
 {
  "id": "S21",
  "name": "lose",
  "music": false,
  "dur": 2.0,
  "prompt": "short gentle sad jingle, soft marimba descending, cute and not dramatic, ends cleanly"
 },
 {
  "id": "S22",
  "name": "music_cafe",
  "music": true,
  "dur": 60.0,
  "prompt": "cozy lo-fi acoustic cafe loop, soft nylon guitar, light brushed drums, warm piano, relaxed, 80 bpm, seamless loop, instrumental"
 },
 {
  "id": "S23",
  "name": "music_level",
  "music": true,
  "dur": 60.0,
  "prompt": "calm thoughtful puzzle game background music, soft marimba and pizzicato strings, light and playful, 90 bpm, seamless loop, instrumental"
 }
]
SEEDS_SFX = 6
SEEDS_MUSIC = 3
NEG_SFX = 'music, speech, voice, reverb tail, background noise, distortion, clipping'
NEG_MUSIC = 'speech, voice, vocals, distortion, clipping, noise'

In [ ]:
import torch, numpy as np, soundfile as sf, os, inspect, csv, datetime, gc
from stable_audio_3 import StableAudioModel
os.makedirs('out/audio', exist_ok=True); os.makedirs('out/candidates', exist_ok=True)

def load(names):
    for n in names:
        try:
            m = StableAudioModel.from_pretrained(n); print('loaded', n); return m, n
        except Exception as e: print('fail', n, e)
    raise RuntimeError(names)

def gen(model, prompt, dur, seed, neg):
    torch.manual_seed(seed); np.random.seed(seed)
    params = inspect.signature(model.generate).parameters
    kw = dict(prompt=prompt, duration=dur)
    if 'seed' in params: kw['seed'] = seed
    if 'negative_prompt' in params: kw['negative_prompt'] = neg
    a = model.generate(**kw)
    sr = getattr(model, 'sample_rate', None) or getattr(model, 'sr', None) or 44100
    if isinstance(a, (tuple, list)):
        if len(a) == 2 and isinstance(a[1], int): a, sr = a
        else: a = a[0]
    if isinstance(a, dict): sr = a.get('sample_rate', sr); a = a.get('audio', a.get('waveform'))
    a = a.detach().float().cpu().numpy() if torch.is_tensor(a) else np.asarray(a, dtype=np.float32)
    a = np.squeeze(a)
    if a.ndim == 2 and a.shape[0] <= 2: a = a.T
    return a, int(sr)

def trim(a, sr, dur, music):
    if music: return a
    n = int((dur + 0.4) * sr)
    mono = np.abs(a if a.ndim == 1 else a.mean(1))
    idx = np.where(mono > mono.max() * 0.03)[0]
    start = max(0, idx[0] - int(0.01 * sr)) if len(idx) else 0
    a = a[start:start + n]
    f = min(len(a), int(0.08 * sr)); ramp = np.linspace(1, 0, f)
    a[-f:] = (a[-f:].T * ramp).T
    return a

In [ ]:
from transformers import ClapModel, ClapProcessor
import librosa
clap = ClapModel.from_pretrained('laion/clap-htsat-unfused').to('cuda').eval()
cproc = ClapProcessor.from_pretrained('laion/clap-htsat-unfused')
def score(a, sr, prompt):
    mono = a if a.ndim == 1 else a.mean(1)
    y = librosa.resample(mono.astype(np.float32), orig_sr=sr, target_sr=48000)
    inp = cproc(text=[prompt], audios=[y], sampling_rate=48000, return_tensors='pt', padding=True).to('cuda')
    with torch.no_grad(): o = clap(**inp)
    clip_pen = float((np.abs(mono) > 0.99).mean()) * 10
    return float(o.logits_per_audio[0, 0]) - clip_pen

In [ ]:
rows = []
def run(group, names, seeds, neg, gen_dur):
    model, mname = load(names)
    for s in group:
        best = []
        for k in range(seeds):
            seed = 1000 + k * 7919
            a, sr = gen(model, s['prompt'], gen_dur(s), seed, neg)
            a = trim(a, sr, s['dur'], s['music'])
            sc = score(a, sr, s['prompt'])
            fn = f"out/candidates/{s['name']}__seed{seed}.wav"; sf.write(fn, a, sr)
            best.append((sc, seed, fn))
        best.sort(reverse=True)
        sc, seed, fn = best[0]
        os.system(f"cp '{fn}' out/audio/{s['name']}.wav")
        rows.append([s['name'] + '.wav', 'Stable Audio 3.0 ' + mname, seed, f'{sc:.2f}', datetime.date.today().isoformat()])
        print(s['id'], s['name'], 'best seed', seed, 'score', round(sc, 2))
    del model; gc.collect(); torch.cuda.empty_cache()

run([s for s in SOUNDS if not s['music']], ['small-sfx'], SEEDS_SFX, NEG_SFX, lambda s: max(2.0, s['dur'] * 2))
run([s for s in SOUNDS if s['music']], ['small-music', 'small', 'medium'], SEEDS_MUSIC, NEG_MUSIC, lambda s: s['dur'])
with open('out/PROVENANCE.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['file', 'model', 'seed', 'clap_score', 'date']); w.writerows(rows)

In [ ]:
!cd out && zip -qr ../kittangle_sounds.zip . && ls -la ..
from google.colab import files
files.download('kittangle_sounds.zip')